# BharatCares: Healthcare Data Analytics with AI
**Internship project submission | Prepared by: Abhishek Kharga**

## Project overview
This notebook demonstrates an end-to-end analytics workflow using a **synthetic, generated healthcare dataset**. It explores patient visit patterns, visualizes key indicators, and trains a simple machine-learning classifier to flag records with a higher likelihood of a follow-up visit.

> **Important:** All records are artificial and generated for educational demonstration. The model is not clinically validated and must not be used for diagnosis, treatment, or real patient decisions.

## 1. Objectives
- Create a reproducible sample dataset without exposing real personal data.
- Clean and inspect the data using pandas.
- Explore patterns in age, wait time, chronic-condition indicators, and follow-up visits.
- Train and evaluate a baseline AI model.
- Summarize limitations and practical next steps.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, ConfusionMatrixDisplay
)

## 2. Generate a synthetic dataset
The data below is simulated locally, so no external dataset download is needed. The target, `follow_up_needed`, is generated from simple artificial patterns and noise; it does not represent real clinical evidence.

In [ ]:
rng = np.random.default_rng(42)
n = 1200

age = rng.integers(18, 90, size=n)
gender = rng.choice(["Female", "Male", "Other"], size=n, p=[0.49, 0.49, 0.02])
department = rng.choice(
    ["General Medicine", "Cardiology", "Orthopedics", "Pediatrics", "Emergency"],
    size=n, p=[0.34, 0.18, 0.17, 0.13, 0.18]
)
wait_minutes = np.clip(rng.gamma(shape=2.5, scale=15, size=n), 2, 180).round().astype(int)
chronic_conditions = np.clip(rng.poisson(lam=np.where(age > 55, 1.5, 0.55)), 0, 5)
previous_visits = np.clip(rng.poisson(lam=np.where(age > 55, 2.2, 1.0)), 0, 12)
satisfaction = np.clip(5 - wait_minutes / 55 + rng.normal(0, 0.7, n), 1, 5).round(1)

logit = (
    -2.0 + 0.025 * (age - 40)
    + 0.60 * chronic_conditions
    + 0.22 * previous_visits
    + 0.25 * (department == "Cardiology")
    + 0.35 * (department == "Emergency")
)
probability = 1 / (1 + np.exp(-logit))
follow_up_needed = rng.binomial(1, probability)

df = pd.DataFrame({
    "age": age,
    "gender": gender,
    "department": department,
    "wait_minutes": wait_minutes,
    "chronic_conditions": chronic_conditions,
    "previous_visits": previous_visits,
    "satisfaction_score": satisfaction,
    "follow_up_needed": follow_up_needed
})

df.head()

## 3. Data quality checks
Review shape, data types, missing values, and duplicates before analysis.

In [ ]:
print("Dataset shape:", df.shape)
display(df.info())
print("\nMissing values by column:\n", df.isna().sum())
print("\nDuplicate rows:", df.duplicated().sum())
display(df.describe(include="all").T)

## 4. Exploratory data analysis
The charts below summarize department volume, waiting time, age distribution, and the generated target balance.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

df["department"].value_counts().plot(kind="bar", ax=axes[0, 0], title="Visits by department")
axes[0, 0].set_xlabel("Department")
axes[0, 0].set_ylabel("Number of visits")
axes[0, 0].tick_params(axis="x", rotation=25)

axes[0, 1].hist(df["wait_minutes"], bins=25)
axes[0, 1].set_title("Waiting time distribution")
axes[0, 1].set_xlabel("Wait (minutes)")
axes[0, 1].set_ylabel("Frequency")

axes[1, 0].hist(df["age"], bins=20)
axes[1, 0].set_title("Age distribution")
axes[1, 0].set_xlabel("Age")
axes[1, 0].set_ylabel("Frequency")

df["follow_up_needed"].value_counts().sort_index().plot(kind="bar", ax=axes[1, 1])
axes[1, 1].set_title("Synthetic follow-up target")
axes[1, 1].set_xlabel("Follow-up flag (0 = no, 1 = yes)")
axes[1, 1].set_ylabel("Records")
axes[1, 1].tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
print("Average waiting time by department:")
display(df.groupby("department")["wait_minutes"].agg(["mean", "median", "count"]).round(1))

print("Follow-up flag rate by department (synthetic target):")
display((df.groupby("department")["follow_up_needed"].mean() * 100).round(1).rename("flag_rate_percent"))

print("Correlation among numeric fields:")
display(df.select_dtypes(include="number").corr().round(2))

## 5. Prepare features and train a baseline AI model
A train/test split is used so evaluation is performed on records the model did not train on. Numeric and categorical columns are preprocessed inside a scikit-learn pipeline.

In [ ]:
target = "follow_up_needed"
X = df.drop(columns=[target])
y = df[target]

numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X.select_dtypes(exclude=["number"]).columns.tolist()

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_pipeline, numeric_features),
    ("cat", categorical_pipeline, categorical_features)
])

model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=150, max_depth=8, min_samples_leaf=4,
        class_weight="balanced", random_state=42
    ))
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)
model.fit(X_train, y_train)
predictions = model.predict(X_test)

metrics = {
    "Accuracy": accuracy_score(y_test, predictions),
    "Precision": precision_score(y_test, predictions, zero_division=0),
    "Recall": recall_score(y_test, predictions, zero_division=0),
    "F1 score": f1_score(y_test, predictions, zero_division=0)
}
pd.DataFrame(metrics.items(), columns=["Metric", "Score"]).round(3)

## 6. Evaluate the model
Accuracy alone can be misleading, so precision, recall, F1-score, and the confusion matrix are also reported. Results are for the synthetic data only.

In [ ]:
print(classification_report(y_test, predictions, zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_test, predictions)
plt.title("Confusion matrix — synthetic demonstration")
plt.show()

## 7. Example prediction
The example is an artificial record. The output is a demonstration of model inference, not a clinical recommendation.

In [ ]:
sample_patient = pd.DataFrame([{
    "age": 67,
    "gender": "Female",
    "department": "General Medicine",
    "wait_minutes": 35,
    "chronic_conditions": 2,
    "previous_visits": 3,
    "satisfaction_score": 3.4
}])

predicted_flag = int(model.predict(sample_patient)[0])
predicted_probability = float(model.predict_proba(sample_patient)[0][1])

print("Predicted synthetic follow-up flag:", predicted_flag)
print("Model-estimated probability:", round(predicted_probability, 3))
print("Interpretation: demonstration output only; not for medical use.")

## 8. Conclusions
- The notebook demonstrates a reproducible data analytics workflow using pandas, NumPy, Matplotlib, and scikit-learn.
- Exploratory charts and grouped summaries help inspect department volume and waiting-time patterns.
- A Random Forest baseline demonstrates a complete model-training and evaluation cycle.
- Since the data and target are synthetic, metrics only indicate that the code runs; they do not establish real-world performance.

## 9. Limitations and future scope
1. Replace synthetic records with an approved, properly de-identified dataset and document its source.
2. Validate target definitions with qualified domain experts before modeling.
3. Compare models and use cross-validation; report confidence intervals and subgroup performance.
4. Review privacy, fairness, and data governance requirements.
5. Treat any future clinical or operational use as a separate, formally validated project.

## Dataset information
No external dataset is used. All data are generated in this notebook with a fixed random seed for reproducibility.